# Lingo Game

Een interactieve implementatie van het klassieke Lingo woordspel.

## Spelregels
- De speler krijgt 5 pogingen om het woord te raden
- Groene achtergrond: letter is correct en op de juiste plaats
- Rode achtergrond: letter zit in het woord maar op de verkeerde plaats
- Geen kleur: letter zit niet in het woord
- Kies tussen 4-letter of 5-letter woorden

## Woordenlijsten

Eerst definiëren we de woordenlijsten voor 4-letter en 5-letter woorden.

In [ ]:
# Woordenlijsten voor het spel

# 50 Nederlandse 4-letter woorden
VIER_LETTER_WOORDEN = [
    'boot', 'kaas', 'lamp', 'ring', 'stok',
    'deur', 'glas', 'boek', 'taak', 'noot',
    'fiets', 'muur', 'duin', 'peer', 'raam',
    'zoon', 'knie', 'beek', 'bank', 'ster',
    'blik', 'hond', 'vuil', 'klok', 'week',
    'melk', 'helm', 'zaak', 'lied', 'kies',
    'gang', 'been', 'blad', 'trap', 'dier',
    'land', 'west', 'boom', 'huis', 'mond',
    'zout', 'hand', 'ogen', 'haar', 'hoek',
    'maag', 'laan', 'maan', 'bord', 'voet'
]

# 50 Nederlandse 5-letter woorden
VIJF_LETTER_WOORDEN = [
    'appel', 'brood', 'krant', 'stoel', 'groep',
    'water', 'licht', 'brief', 'toren', 'strand',
    'plant', 'schip', 'vogel', 'macht', 'paard',
    'zwart', 'groen', 'steen', 'graag', 'klein',
    'groot', 'snaar', 'vloer', 'bezem', 'trein',
    'koets', 'worst', 'baker', 'beker', 'kroon',
    'fruit', 'zacht', 'vloot', 'geest', 'boter',
    'vlees', 'kaart', 'rozen', 'pijns', 'storm',
    'kracht', 'kunst', 'datum', 'onder', 'vrede',
    'welke', 'wijze', 'zalig', 'broek', 'jurks'
]

## Hulpfuncties

Functies voor het weergeven van gekleurde tekst in de terminal.

In [ ]:
# ANSI kleurcodes voor terminal output
def kleur_groen(tekst):
    """Geef tekst weer met groene achtergrond (correcte positie)"""
    return f"\033[42m\033[30m {tekst} \033[0m"

def kleur_rood(tekst):
    """Geef tekst weer met rode achtergrond (verkeerde positie)"""
    return f"\033[41m\033[37m {tekst} \033[0m"

def kleur_grijs(tekst):
    """Geef tekst weer met grijze achtergrond (letter niet in woord)"""
    return f"\033[47m\033[30m {tekst} \033[0m"

def toon_feedback(woord, gok):
    """
    Toon de gok met kleurgecodeerde feedback.
    
    Parameters:
        woord (str): Het te raden woord
        gok (str): De gok van de speler
    
    Returns:
        str: Gekleurde string met feedback
    """
    resultaat = []
    woord_letters = list(woord)
    gok_letters = list(gok)
    
    # Markeer eerst alle correcte posities (groen)
    correct_posities = []
    for i in range(len(gok_letters)):
        if gok_letters[i] == woord_letters[i]:
            correct_posities.append(i)
    
    # Maak een lijst van beschikbare letters (exclusief correcte posities)
    beschikbare_letters = []
    for i, letter in enumerate(woord_letters):
        if i not in correct_posities:
            beschikbare_letters.append(letter)
    
    # Bouw het resultaat op
    for i, letter in enumerate(gok_letters):
        if i in correct_posities:
            # Correcte positie (groen)
            resultaat.append(kleur_groen(letter.upper()))
        elif letter in beschikbare_letters:
            # Letter zit in woord maar verkeerde positie (rood)
            resultaat.append(kleur_rood(letter.upper()))
            # Verwijder de letter uit beschikbare letters (elke letter maar 1x tellen)
            beschikbare_letters.remove(letter)
        else:
            # Letter zit niet in woord (grijs)
            resultaat.append(kleur_grijs(letter.upper()))
    
    return ''.join(resultaat)

## Validatiefuncties

Functies voor het valideren van invoer.

In [ ]:
def valideer_gok(gok, lengte):
    """
    Valideer of de gok geldig is.
    
    Parameters:
        gok (str): De gok van de speler
        lengte (int): Vereiste lengte van het woord
    
    Returns:
        tuple: (is_geldig, foutmelding)
    """
    if not gok:
        return False, "Je moet een woord invoeren!"
    
    if len(gok) != lengte:
        return False, f"Het woord moet precies {lengte} letters hebben!"
    
    if not gok.isalpha():
        return False, "Het woord mag alleen letters bevatten!"
    
    return True, ""

## Hoofdspel Functie

De hoofdfunctie die het spel bestuurt.

In [ ]:
import random

def speel_lingo():
    """
    Hoofdfunctie voor het Lingo spel.
    """
    print("="*60)
    print("🎮 WELKOM BIJ LINGO! 🎮")
    print("="*60)
    print()
    print("Spelregels:")
    print("- Je hebt 5 pogingen om het woord te raden")
    print("- Groene achtergrond = letter is correct en op de juiste plaats")
    print("- Rode achtergrond = letter zit in het woord maar verkeerde plaats")
    print("- Grijze achtergrond = letter zit niet in het woord")
    print()
    
    # Kies woordlengte
    while True:
        keuze = input("Wil je 4-letter of 5-letter woorden spelen? (4/5): ").strip()
        if keuze == '4':
            woordenlijst = VIER_LETTER_WOORDEN
            woordlengte = 4
            break
        elif keuze == '5':
            woordenlijst = VIJF_LETTER_WOORDEN
            woordlengte = 5
            break
        else:
            print("Ongeldige keuze! Kies 4 of 5.")
    
    # Kies willekeurig woord
    te_raden_woord = random.choice(woordenlijst).lower()
    max_pogingen = 5
    poging_nummer = 0
    
    print(f"\n🎯 Het spel begint! Raad het {woordlengte}-letter woord.")
    print(f"Eerste letter: {te_raden_woord[0].upper()}")
    print("="*60)
    
    # Spel loop
    while poging_nummer < max_pogingen:
        poging_nummer += 1
        print(f"\nPoging {poging_nummer}/{max_pogingen}")
        
        # Vraag om invoer
        gok = input("Jouw gok: ").strip().lower()
        
        # Valideer invoer
        is_geldig, foutmelding = valideer_gok(gok, woordlengte)
        if not is_geldig:
            print(f"❌ {foutmelding}")
            poging_nummer -= 1  # Tel deze poging niet mee
            continue
        
        # Toon feedback
        feedback = toon_feedback(te_raden_woord, gok)
        print(f"   {feedback}")
        
        # Check of woord correct is
        if gok == te_raden_woord:
            print("="*60)
            print("🎉 GEFELICITEERD! Je hebt het woord geraden! 🎉")
            print(f"Het woord was: {te_raden_woord.upper()}")
            print(f"Je hebt {poging_nummer} {'poging' if poging_nummer == 1 else 'pogingen'} nodig gehad.")
            print("="*60)
            return True
    
    # Alle pogingen op
    print("="*60)
    print("😞 Helaas! Je hebt geen pogingen meer.")
    print(f"Het woord was: {te_raden_woord.upper()}")
    print("="*60)
    return False

## Spel Loop

Functie om meerdere rondes te spelen.

In [ ]:
def speel_meerdere_rondes():
    """
    Speel meerdere rondes Lingo.
    """
    rondes_gespeeld = 0
    rondes_gewonnen = 0
    
    while True:
        # Speel een ronde
        rondes_gespeeld += 1
        gewonnen = speel_lingo()
        
        if gewonnen:
            rondes_gewonnen += 1
        
        # Vraag of speler nog een ronde wil spelen
        print(f"\n📊 Score: {rondes_gewonnen}/{rondes_gespeeld} gewonnen")
        nog_een_keer = input("\nWil je nog een ronde spelen? (j/n): ").strip().lower()
        
        if nog_een_keer != 'j' and nog_een_keer != 'ja':
            print("\n" + "="*60)
            print("👋 Bedankt voor het spelen!")
            print(f"Eindstand: {rondes_gewonnen}/{rondes_gespeeld} gewonnen")
            if rondes_gespeeld > 0:
                percentage = (rondes_gewonnen / rondes_gespeeld) * 100
                print(f"Winpercentage: {percentage:.1f}%")
            print("="*60)
            break
        
        print("\n" + "\n")  # Extra ruimte tussen rondes

## Start het Spel!

Run de onderstaande cel om het spel te starten.

In [ ]:
# Start het spel
speel_meerdere_rondes()

## Test Functie (Optioneel)

Voor testdoeleinden: toon hoe de feedback werkt.

In [ ]:
def test_feedback():
    """
    Test de feedback functie met voorbeelden.
    """
    print("Test Feedback Systeem")
    print("="*60)
    
    test_gevallen = [
        ('appel', 'appel', 'Perfecte match'),
        ('appel', 'pleap', 'Alle letters verkeerd geplaatst'),
        ('appel', 'boter', 'Alleen E correct geplaatst'),
        ('kaas', 'maas', 'Laatste 3 letters correct'),
        ('boek', 'krab', 'Geen enkele letter correct'),
    ]
    
    for woord, gok, beschrijving in test_gevallen:
        print(f"\nWoord: {woord.upper()}")
        print(f"Gok:   {gok.upper()}")
        print(f"Info:  {beschrijving}")
        print(f"Result: {toon_feedback(woord, gok)}")
    
    print("\n" + "="*60)

# Uncomment de volgende regel om de test uit te voeren
# test_feedback()